In [ ]:
"""
Manure Manifest Geocode Review

Manual review of the geocoded origin and destination points and the routed
paths in processed_manure_manifests_analysis.geojson (the tool in Figure S3).

Workflow (cells are labelled by their section banner):
  1. Run down to and including "3) GENERATE THE HTML REVIEWER", which writes
     the reviewer to REVIEW_HTML_FILE.
  2. Open it in a browser. Each screen shows the origin dairy, the destination
     and the whole route on satellite imagery. Judge each endpoint with the
     buttons or number keys; key 5 flags a bad route whose endpoints are right.
  3. Press "Link CSV" and choose the existing geocode_review.csv; every verdict
     is then written to that file as you click (Chromium browsers only).
     Otherwise press "Export decisions" and run section 4. "New CSV" creates a
     side file when none exists and is the only button that can overwrite a
     file. Progress is also kept in browser storage.
  4. Run "5) APPLY THE VERDICTS TO THE ANALYSIS AND OD FILES". A verdict marked
     wrong sets valid route = False with the reason "Bad Origin Geocode",
     "Bad Destination Geocode" or "Routing Incorrect", which removes the
     manifest from the routing results.

Review is per unique point: a dairy or destination that appears on many
manifests is judged once. Verdicts are keyed by address and coordinates, so
after a new routing run every unchanged point keeps its verdict.
"""
import os
import json
import html

import numpy as np
import pandas as pd
import geopandas as gpd

In [ ]:
# =========================================================================
# 0) FILE PATHS
# =========================================================================
# Paths are relative to the repository root, found by walking up from the
# working directory.
def find_repo_root():
    """Nearest folder at or above the working directory holding data/ and
    analysis/Analysis.ipynb."""
    d = os.path.abspath(os.getcwd())
    while True:
        if (os.path.isdir(os.path.join(d, "data"))
                and os.path.isfile(os.path.join(d, "analysis", "Analysis.ipynb"))):
            return d + os.sep
        if os.path.dirname(d) == d:
            raise FileNotFoundError(
                f"no repository root above {os.getcwd()}; run this notebook "
                "from inside the manure-trucking folder")
        d = os.path.dirname(d)


ROOT_PATH = find_repo_root()

# Written by analysis/Routing.ipynb; section 5 rewrites them in place.
MANURE_ANALYSIS_FILE = ROOT_PATH + "outputs/results/routing/processed_manure_manifests_analysis.geojson"
OD_LINES_FILE        = ROOT_PATH + "outputs/results/routing/od_lines_with_status.geojson"
OD_PAIRS_FILE        = ROOT_PATH + "outputs/results/routing/od_pairs_with_status.geojson"

# The manual verdicts, keyed by point so they survive a new routing run.
GEOCODE_REVIEW_FILE  = ROOT_PATH + "outputs/review/manifests/geocode_review.csv"
REVIEW_HTML_FILE     = ROOT_PATH + "review/manifests/geocode-review/geocode_review.html"
REVIEW_EXPORT_FILE   = ROOT_PATH + "review/manifests/geocode-review/geocode_review_export.csv"

# =========================================================================
# COLUMN NAMES  (must match analysis/Routing.ipynb)
# =========================================================================
ORIG_LAT_COL  = "Origin Dairy Latitude (Geocoded)"
ORIG_LON_COL  = "Origin Dairy Longitude (Geocoded)"
DEST_LAT_COL  = "Destination Latitude (Geocoded)"
DEST_LON_COL  = "Destination Longitude (Geocoded)"
ORIG_NAME_COL = "Origin Dairy Name"
ORIG_ADDR_COL = "Origin Dairy Address"
DEST_ADDR_COL = "Destination Address Final"
DEST_TYPE_COL = "Destination Type (Standardized)"
ORIG_OK_COL   = "Origin Geocode Correct"
DEST_OK_COL   = "Destination Geocode Correct"
ROUTE_OK_COL  = "Route Correct"
VALID_COL     = "valid route"
REASON_COL    = "validation reason"

# Validation reasons set by this notebook, applied in this order to manifests
# that routing left valid (the review drops in Figure S6).
REASON_BAD_ORIGIN  = "Bad Origin Geocode"
REASON_BAD_DEST    = "Bad Destination Geocode"
REASON_ROUTE_WRONG = "Routing Incorrect"

# Simplification tolerance for the route lines in the HTML, in degrees
# (0.001 is about 100 m).
ROUTE_SIMPLIFY_DEG = 0.001
INCLUDE_ROUTE_GEOMETRY = True

In [ ]:
# =========================================================================
# 1) HELPERS
# =========================================================================
def geocode_key(side, address, lat, lon):
    """Stable identity for one geocoded point: side + tidied address + coords.

    Must match tidyKey() in the reviewer JavaScript, or verdicts will not
    re-join.
    """
    if pd.isna(lat) or pd.isna(lon):
        return None
    addr = " ".join(str(address).strip().lower().split()) if pd.notna(address) else ""
    return f"{side}|{addr}|{float(lat):.6f}|{float(lon):.6f}"

def route_key(olat, olon, dlat, dlon):
    """Stable identity for one routed path: both endpoint coordinates.

    A shortest path is fixed by its endpoints, so every manifest sharing an
    origin and destination shares this key and this verdict.
    """
    vals = [olat, olon, dlat, dlon]
    if any(v is None or pd.isna(v) for v in vals):
        return None
    return (f"route|{float(olat):.6f},{float(olon):.6f}"
            f"|{float(dlat):.6f},{float(dlon):.6f}")

def row_key(row):
    """Key for one side-file row: the stored key, else rebuilt from the point."""
    k = row.get("key")
    if k is not None and pd.notna(k) and str(k).strip():
        return str(k)
    return geocode_key(row["side"], row["address"],
                       pd.to_numeric(row["lat"], errors="coerce"),
                       pd.to_numeric(row["lon"], errors="coerce"))

def load_review(review_file):
    """Existing verdicts as {key: verdict}, empty if nothing has been reviewed."""
    if not os.path.exists(review_file):
        print(f"No review file yet at {review_file}")
        return {}
    df = pd.read_csv(review_file)
    out = {}
    for _, r in df.iterrows():
        k = row_key(r)
        if k is not None:
            out[k] = r["verdict"]
    n_route = sum(1 for k in out if k.startswith("route|"))
    print(f"Loaded {len(out):,} existing verdicts from {os.path.basename(review_file)}"
          f" ({n_route:,} route flags)")
    return out

def save_review(verdict_rows, review_file):
    """Write the side file, keeping the newest verdict per key."""
    cols = ["key", "side", "address", "lat", "lon", "verdict", "reviewed_utc"]
    df = pd.DataFrame(verdict_rows)
    for c in cols:
        if c not in df.columns:
            df[c] = pd.NA
    df = df[cols]
    df["key"] = [row_key(r) for _, r in df.iterrows()]
    df = df[df["key"].notna()].drop_duplicates(subset="key", keep="last")
    os.makedirs(os.path.dirname(review_file), exist_ok=True)
    df.to_csv(review_file, index=False)
    print(f"Wrote {len(df):,} verdicts to {review_file}")
    return df

def as_verdict(value):
    """Normalise a stored verdict to True, False or None."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    s = str(value).strip().lower()
    if s in ("true", "t", "yes", "y", "1", "1.0"):
        return True
    if s in ("false", "f", "no", "n", "0", "0.0"):
        return False
    return None

In [ ]:
# =========================================================================
# 2) BUILD THE REVIEW QUEUE
# =========================================================================
print("Loading manifests...")
if not os.path.exists(MANURE_ANALYSIS_FILE):
    raise FileNotFoundError(
        f"{MANURE_ANALYSIS_FILE} not found.\n"
        "  Run analysis/Routing.ipynb first to generate it."
    )
manifests = gpd.read_file(MANURE_ANALYSIS_FILE)
required = ["RouteID", ORIG_LAT_COL, ORIG_LON_COL, DEST_LAT_COL, DEST_LON_COL]
missing = [c for c in required if c not in manifests.columns]
if missing:
    raise KeyError(
        f"{os.path.basename(MANURE_ANALYSIS_FILE)} is missing {missing}. "
        "Re-run analysis/Routing.ipynb - older copies of this file "
        "predate the RouteID column."
    )
print(f"  {len(manifests):,} manifests")

olat = pd.to_numeric(manifests[ORIG_LAT_COL], errors="coerce")
olon = pd.to_numeric(manifests[ORIG_LON_COL], errors="coerce")
dlat = pd.to_numeric(manifests[DEST_LAT_COL], errors="coerce")
dlon = pd.to_numeric(manifests[DEST_LON_COL], errors="coerce")
oaddr = manifests[ORIG_ADDR_COL] if ORIG_ADDR_COL in manifests.columns else pd.Series("", index=manifests.index)
daddr = manifests[DEST_ADDR_COL] if DEST_ADDR_COL in manifests.columns else pd.Series("", index=manifests.index)

manifests["_okey"] = [geocode_key("origin", a, y, x) for a, y, x in zip(oaddr, olat, olon)]
manifests["_dkey"] = [geocode_key("destination", a, y, x) for a, y, x in zip(daddr, dlat, dlon)]

# Route geometry for the overview panel, keyed by RouteID
routes = {}
if INCLUDE_ROUTE_GEOMETRY and os.path.exists(OD_LINES_FILE):
    od_lines = gpd.read_file(OD_LINES_FILE)
    ok = od_lines.geometry.notna()
    simp = od_lines[ok].copy()
    simp["geometry"] = simp.geometry.simplify(ROUTE_SIMPLIFY_DEG, preserve_topology=False)
    for rid, geom in zip(simp["RouteID"], simp.geometry):
        if geom is None or geom.is_empty:
            continue
        if geom.geom_type == "LineString":
            coords = list(geom.coords)
        elif hasattr(geom, "geoms"):
            coords = [c for part in geom.geoms for c in part.coords]
        else:
            continue
        if len(coords) < 2:
            continue
        routes[int(rid)] = [[round(y, 5), round(x, 5)] for x, y in coords]
    print(f"  {len(routes):,} routed lines available for the overview panel")
else:
    print("  No routed lines found; the overview panel will draw a straight connector")

# One screen per unique (origin point, destination point) combination
existing = load_review(GEOCODE_REVIEW_FILE)
queue, seen = [], set()
for _, row in manifests.iterrows():
    combo = (row["_okey"], row["_dkey"])
    if combo in seen or (row["_okey"] is None and row["_dkey"] is None):
        continue
    seen.add(combo)
    rid = int(row["RouteID"])
    queue.append({
        "route_id": rid,
        "okey": row["_okey"], "dkey": row["_dkey"],
        "oname": ("" if pd.isna(row.get(ORIG_NAME_COL)) else str(row.get(ORIG_NAME_COL))),
        "oaddr": ("" if pd.isna(row.get(ORIG_ADDR_COL)) else str(row.get(ORIG_ADDR_COL))),
        "daddr": ("" if pd.isna(row.get(DEST_ADDR_COL)) else str(row.get(DEST_ADDR_COL))),
        "dtype": ("" if pd.isna(row.get(DEST_TYPE_COL)) else str(row.get(DEST_TYPE_COL))),
        "olat": None if pd.isna(olat[row.name]) else round(float(olat[row.name]), 6),
        "olon": None if pd.isna(olon[row.name]) else round(float(olon[row.name]), 6),
        "dlat": None if pd.isna(dlat[row.name]) else round(float(dlat[row.name]), 6),
        "dlon": None if pd.isna(dlon[row.name]) else round(float(dlon[row.name]), 6),
        "route": routes.get(rid),
        "rkey": route_key(olat[row.name], olon[row.name], dlat[row.name], dlon[row.name]),
    })

n_o = len({q["okey"] for q in queue if q["okey"]})
n_d = len({q["dkey"] for q in queue if q["dkey"]})
print(f"\n  {len(queue):,} screens covering {len(manifests):,} manifests")
print(f"  {n_o:,} unique origin points and {n_d:,} unique destination points to judge")
print(f"  already decided: {sum(1 for k in existing if k.startswith('origin|')):,} origins, "
      f"{sum(1 for k in existing if k.startswith('destination|')):,} destinations, "
      f"{sum(1 for k in existing if k.startswith('route|')):,} route flags")

In [ ]:
# =========================================================================
# 3) GENERATE THE HTML REVIEWER
# =========================================================================
REVIEW_HTML_TEMPLATE = r"""<!DOCTYPE html>
<html><head><meta charset="utf-8"><title>Manure geocode review</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<style>
 body{font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;margin:0;background:#1b1b1b;color:#eee}
 header{padding:10px 16px;background:#111;border-bottom:1px solid #333;display:flex;
        gap:18px;align-items:center;flex-wrap:wrap}
 .maps{display:grid;grid-template-columns:1fr 1fr 1fr;gap:8px;padding:8px}
 .panel{background:#222;border:1px solid #333;border-radius:6px;overflow:hidden}
 .panel h3{margin:0;padding:7px 10px;font-size:13px;background:#2a2a2a;border-bottom:1px solid #333}
 .meta{padding:6px 10px;font-size:12px;color:#bbb;height:44px;overflow:hidden}
 .map{height:60vh}
 .verdict{padding:8px 10px;display:flex;gap:8px;align-items:center;border-top:1px solid #333}
 button{cursor:pointer;border:1px solid #555;background:#333;color:#eee;border-radius:4px;
        padding:6px 12px;font-size:13px}
 button:hover{background:#444}
 .ok.on{background:#2ca02c;border-color:#2ca02c;color:#fff}
 .bad.on{background:#d62728;border-color:#d62728;color:#fff}
 .pill{padding:3px 9px;border-radius:10px;background:#333;font-size:12px}
 kbd{background:#333;border:1px solid #555;border-radius:3px;padding:1px 5px;font-size:11px}
 #done{color:#2ca02c}
 .panel.settled{opacity:.4;transition:opacity .12s}
 .panel.settled:hover{opacity:1}
 .panel.settled h3::after{content:" \2014 decided";color:#9a9a9a;font-weight:400}
</style></head><body>
<header>
  <button onclick="go(-1)">&larr; Prev <kbd>&larr;</kbd></button>
  <button onclick="go(1)">Next <kbd>&rarr;</kbd></button>
  <span class="pill">Screen <b id="pos"></b> / <b id="total"></b></span>
  <span class="pill">RouteID <b id="rid"></b></span>
  <span class="pill">Origins <b id="oprog"></b></span>
  <span class="pill">Destinations <b id="dprog"></b></span>
  <span class="pill">Routes flagged <b id="rprog"></b></span>
  <span class="pill">Screens left <b id="sleft"></b></span>
  <label style="font-size:12px"><input type="checkbox" id="skipDone" checked> skip decided</label>
  <button id="linkBtn" onclick="linkFile()">Link CSV</button>
  <button id="newBtn" onclick="newFile()">New CSV…</button>
  <button id="grantBtn" onclick="grant()" style="display:none">Reconnect file</button>
  <span class="pill" id="fsStatus">not linked</span>
  <button onclick="exportCsv()">Export decisions</button>
  <span id="done"></span>
</header>
<div class="maps">
  <div class="panel" id="opanel"><h3>Origin dairy</h3><div class="meta" id="ometa"></div>
    <div class="map" id="omap"></div>
    <div class="verdict"><button class="ok"  id="obtn1" onclick="setV('o',true)">Correct <kbd>1</kbd></button>
      <button class="bad" id="obtn0" onclick="setV('o',false)">Wrong <kbd>2</kbd></button>
      <button onclick="setV('o',null)">Clear</button></div></div>
  <div class="panel" id="dpanel"><h3>Destination</h3><div class="meta" id="dmeta"></div>
    <div class="map" id="dmap"></div>
    <div class="verdict"><button class="ok"  id="dbtn1" onclick="setV('d',true)">Correct <kbd>3</kbd></button>
      <button class="bad" id="dbtn0" onclick="setV('d',false)">Wrong <kbd>4</kbd></button>
      <button onclick="setV('d',null)">Clear</button></div></div>
  <div class="panel" id="rpanel"><h3>Whole route</h3><div class="meta" id="rmeta"></div>
    <div class="map" id="rmap"></div>
    <div class="verdict"><button class="bad" id="rbtn0" onclick="setV('r',false)">Routing failed <kbd>5</kbd></button>
      <button class="ok" id="rbtn1" onclick="setV('r',true)">Route OK <kbd>6</kbd></button>
      <button onclick="setV('r',null)">Clear</button></div></div>
</div>
<script>
const ITEMS = __ITEMS__;
const SEED  = __SEED__;
const STORE = "manure_geocode_review_v1";
let V  = Object.assign({}, SEED, JSON.parse(localStorage.getItem(STORE) || "{}"));
let ST = JSON.parse(localStorage.getItem(STORE + "_ts") || "{}");   // key -> reviewed_utc
let i = 0;

// Untidied address and coordinates per key, so the CSV keeps readable
// addresses. For points no longer in the manifests they come from the key.
const META = {};
for(const it of ITEMS){
  if(it.okey) META[it.okey] = {side:"origin",      address:it.oaddr, lat:it.olat, lon:it.olon};
  if(it.dkey) META[it.dkey] = {side:"destination", address:it.daddr, lat:it.dlat, lon:it.dlon};
  if(it.rkey) META[it.rkey] = {side:"route", address:(it.oaddr + " -> " + it.daddr),
                               lat:it.olat, lon:it.olon};
}
function keyMeta(k){
  const a = k.indexOf("|"), c = k.lastIndexOf("|"), b = k.lastIndexOf("|", c - 1);
  return {side:k.slice(0,a), address:k.slice(a+1,b), lat:k.slice(b+1,c), lon:k.slice(c+1)};
}
function tidyKey(side, address, lat, lon){
  const a = String(address == null ? "" : address).trim().toLowerCase().replace(/\s+/g, " ");
  return side + "|" + a + "|" + Number(lat).toFixed(6) + "|" + Number(lon).toFixed(6);
}
function persistLocal(){
  localStorage.setItem(STORE, JSON.stringify(V));
  localStorage.setItem(STORE + "_ts", JSON.stringify(ST));
}

function base(id){
  const m = L.map(id, {zoomControl:true});
  L.tileLayer("https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    {maxZoom:19, attribution:"Imagery &copy; Esri"}).addTo(m);
  L.tileLayer("https://server.arcgisonline.com/ArcGIS/rest/services/Reference/World_Transportation/MapServer/tile/{z}/{y}/{x}",
    {maxZoom:19}).addTo(m);
  L.tileLayer("https://server.arcgisonline.com/ArcGIS/rest/services/Reference/World_Boundaries_and_Places/MapServer/tile/{z}/{y}/{x}",
    {maxZoom:19}).addTo(m);
  m.setView([36.7,-119.8], 8);
  return m;
}
const omap = base("omap"), dmap = base("dmap"), rmap = base("rmap");
let layers = [];
function clear(){ layers.forEach(l => l.remove()); layers = []; }
function dot(map, lat, lon, color, label){
  const mk = L.circleMarker([lat,lon], {radius:9,color:"#fff",weight:2,
              fillColor:color,fillOpacity:0.95}).addTo(map);
  if(label) mk.bindTooltip(label, {permanent:false});
  layers.push(mk); return mk;
}
function esc(s){ return (s||"").replace(/[&<>]/g, c => ({"&":"&amp;","<":"&lt;",">":"&gt;"}[c])); }

function render(){
  const it = ITEMS[i];
  clear();
  document.getElementById("pos").textContent = i+1;
  document.getElementById("total").textContent = ITEMS.length;
  document.getElementById("rid").textContent = it.route_id;
  document.getElementById("ometa").innerHTML =
    "<b>"+esc(it.oname)+"</b><br>"+esc(it.oaddr) + (it.olat===null ? " <i>(no geocode)</i>" : "");
  document.getElementById("dmeta").innerHTML =
    "<b>"+esc(it.dtype)+"</b><br>"+esc(it.daddr) + (it.dlat===null ? " <i>(no geocode)</i>" : "");

  if(it.olat!==null){ omap.setView([it.olat,it.olon], 17); dot(omap,it.olat,it.olon,"#1f77b4",esc(it.oname)); }
  if(it.dlat!==null){ dmap.setView([it.dlat,it.dlon], 17); dot(dmap,it.dlat,it.dlon,"#ff7f0e",esc(it.daddr)); }

  const pts = [];
  if(it.olat!==null){ dot(rmap,it.olat,it.olon,"#1f77b4"); pts.push([it.olat,it.olon]); }
  if(it.dlat!==null){ dot(rmap,it.dlat,it.dlon,"#ff7f0e"); pts.push([it.dlat,it.dlon]); }
  if(it.route){
    const pl = L.polyline(it.route, {color:"#ffd400",weight:4,opacity:0.9}).addTo(rmap);
    layers.push(pl); it.route.forEach(p => pts.push(p));
    document.getElementById("rmeta").textContent = "Routed path, " + it.route.length + " vertices";
  } else if(pts.length===2){
    const pl = L.polyline(pts, {color:"#ffd400",weight:3,dashArray:"6,6",opacity:0.8}).addTo(rmap);
    layers.push(pl);
    document.getElementById("rmeta").textContent = "No routed path - straight connector shown";
  } else {
    document.getElementById("rmeta").textContent = "Only one endpoint available";
  }
  if(pts.length) rmap.fitBounds(L.latLngBounds(pts).pad(0.35));

  paint();
}
function paint(){
  const it = ITEMS[i];
  for(const [side,key] of [["o",it.okey],["d",it.dkey],["r",it.rkey]]){
    const v = key ? V[key] : undefined;
    document.getElementById(side+"btn1").classList.toggle("on", v===true);
    document.getElementById(side+"btn0").classList.toggle("on", v===false);
    document.getElementById(side+"btn1").disabled = !key;
    document.getElementById(side+"btn0").disabled = !key;
    // Dim panels that already have a verdict; hovering restores them, and
    // their buttons stay active so a verdict can be changed.
    document.getElementById(side+"panel").classList.toggle("settled", v !== undefined);
  }
  const oKeys = new Set(ITEMS.map(x=>x.okey).filter(Boolean));
  const dKeys = new Set(ITEMS.map(x=>x.dkey).filter(Boolean));
  const nO = [...oKeys].filter(k=>k in V).length, nD = [...dKeys].filter(k=>k in V).length;
  const nR = Object.keys(V).filter(k => k.startsWith("route|") && V[k] === false).length;
  document.getElementById("oprog").textContent = nO+" / "+oKeys.size;
  document.getElementById("dprog").textContent = nD+" / "+dKeys.size;
  document.getElementById("rprog").textContent = nR;
  document.getElementById("sleft").textContent =
    ITEMS.filter(x => !decided(x)).length + " / " + ITEMS.length;
  document.getElementById("done").textContent =
    (nO===oKeys.size && nD===dKeys.size) ? "all reviewed - export now" : "";
}
function setV(side, val){
  const it = ITEMS[i];
  const key = side==="o" ? it.okey : (side==="d" ? it.dkey : it.rkey);
  if(!key) return;
  if(val===null){ delete V[key]; delete ST[key]; }
  else { V[key] = val; ST[key] = new Date().toISOString(); }
  persistLocal();
  scheduleSave();
  paint();
  if(side === "r") return;
  const it2 = ITEMS[i];
  const oDone = !it2.okey || (it2.okey in V), dDone = !it2.dkey || (it2.dkey in V);
  if(oDone && dDone && document.getElementById("skipDone").checked) setTimeout(()=>go(1), 220);
}
function decided(it){
  const o = !it.okey || (it.okey in V), d = !it.dkey || (it.dkey in V);
  return o && d;
}
function go(step){
  const skip = document.getElementById("skipDone").checked;
  let j = i;
  for(let n=0; n<ITEMS.length; n++){
    j = (j + step + ITEMS.length) % ITEMS.length;
    if(!skip || !decided(ITEMS[j])) { i = j; render(); return; }
  }
  i = (i + step + ITEMS.length) % ITEMS.length; render();
}
function buildCsv(){
  const rows = [["key","side","address","lat","lon","verdict","reviewed_utc"]];
  const now = new Date().toISOString();
  for(const key of Object.keys(V).sort()){
    const m = META[key] || keyMeta(key);
    rows.push([key, m.side, m.address, m.lat, m.lon, V[key] ? "True" : "False", ST[key] || now]);
  }
  return rows.map(r => r.map(c => '"'+String(c).replace(/"/g,'""')+'"').join(",")).join("\n");
}
function exportCsv(){
  const a = document.createElement("a");
  a.href = URL.createObjectURL(new Blob([buildCsv()], {type:"text/csv"}));
  a.download = "geocode_review_export.csv"; a.click();
}
function parseCsv(text){
  const rows = [];
  for(const line of text.split(/\r?\n/)){
    if(!line.trim()) continue;
    const cells = []; let cur = "", q = false;
    for(let c = 0; c < line.length; c++){
      const ch = line[c];
      if(q){
        if(ch === '"'){ if(line[c+1] === '"'){ cur += '"'; c++; } else q = false; }
        else cur += ch;
      } else if(ch === '"') q = true;
      else if(ch === ","){ cells.push(cur); cur = ""; }
      else cur += ch;
    }
    cells.push(cur);
    if(cells.length >= 5) rows.push(cells);
  }
  return rows.slice(1);   // drop the header
}

// ---- write straight to geocode_review.csv (File System Access API) ----
const HAS_FS = (typeof window.showSaveFilePicker === "function");
let fileHandle = null, saveTimer = null;

function idb(){
  return new Promise((res, rej) => {
    const r = indexedDB.open("manure_review_fs", 1);
    r.onupgradeneeded = () => r.result.createObjectStore("h");
    r.onsuccess = () => res(r.result);
    r.onerror  = () => rej(r.error);
  });
}
async function idbPut(k, v){
  const d = await idb();
  return new Promise((res, rej) => {
    const t = d.transaction("h", "readwrite");
    t.objectStore("h").put(v, k);
    t.oncomplete = () => res(); t.onerror = () => rej(t.error);
  });
}
async function idbGet(k){
  const d = await idb();
  return new Promise((res, rej) => {
    const t = d.transaction("h", "readonly");
    const q = t.objectStore("h").get(k);
    q.onsuccess = () => res(q.result); q.onerror = () => rej(q.error);
  });
}
function fsStatus(text, colour){
  const el = document.getElementById("fsStatus");
  el.textContent = text;
  el.style.background = colour || "#333";
  el.style.color = colour ? "#fff" : "#eee";
}
async function loadFromFile(){
  // Merge the verdicts already in the file before writing.
  const f = await fileHandle.getFile();
  const text = await f.text();
  let added = 0;
  const header = (text.split(/\r?\n/)[0] || "").toLowerCase();
  const hasKey = header.indexOf('"key"') === 0 || header.indexOf("key,") === 0;
  for(const c of parseCsv(text)){
    const [side, address, lat, lon, verdict, when] = hasKey ? c.slice(1) : c;
    const k = hasKey ? c[0] : ((lat === "" || lon === "") ? null : tidyKey(side, address, lat, lon));
    if(!k) continue;
    if(!(k in V)){
      V[k] = /^(true|t|yes|y|1|1\.0)$/i.test(String(verdict).trim());
      ST[k] = when || "";
      if(!META[k]) META[k] = {side, address, lat, lon};
      added++;
    }
  }
  if(added) { persistLocal(); paint(); }
  return added;
}
async function saveNow(){
  if(!fileHandle) return;
  try{
    const w = await fileHandle.createWritable();
    await w.write(buildCsv());
    await w.close();
    fsStatus("saved " + new Date().toLocaleTimeString(), "#2ca02c");
  }catch(e){
    fsStatus("save failed: " + e.message, "#d62728");
  }
}
function scheduleSave(){
  if(!fileHandle) return;
  fsStatus("saving...", "#8a6d00");
  clearTimeout(saveTimer);
  saveTimer = setTimeout(saveNow, 600);
}
// Shared by both buttons: request readwrite access, check the file is a
// side file, merge its verdicts, then write the merged set back.
async function attach(h){
  fileHandle = h;
  let p = await fileHandle.queryPermission({mode:"readwrite"});
  if(p !== "granted") p = await fileHandle.requestPermission({mode:"readwrite"});
  if(p !== "granted"){
    fileHandle = null;
    fsStatus("permission denied", "#d62728");
    return;
  }
  // Saving replaces the whole file, so a file that is not a side file is refused.
  const text = await (await fileHandle.getFile()).text();
  const hdr = (text.split(/\r?\n/)[0] || "").toLowerCase();
  const isSide = hdr.indexOf("key,") === 0 || hdr.indexOf('"key"') === 0
              || hdr.indexOf("side,") === 0 || hdr.indexOf('"side"') === 0;
  if(text.trim() && !isSide){
    fileHandle = null;
    fsStatus("not a review file - left untouched", "#d62728");
    return;
  }
  const added = await loadFromFile();
  await idbPut("handle", fileHandle);
  await saveNow();
  document.getElementById("grantBtn").style.display = "none";
  fsStatus("linked" + (added ? ", " + added + " existing verdicts loaded" : ""),
           "#2ca02c");
}
async function linkFile(){
  if(!HAS_FS){ fsStatus("browser cannot write files - use Export", "#d62728"); return; }
  if(typeof window.showOpenFilePicker !== "function"){
    fsStatus("browser cannot open files - use New CSV", "#d62728");
    return;
  }
  try{
    // showOpenFilePicker opens an existing file; showSaveFilePicker (used by
    // newFile) can replace the chosen file.
    const [h] = await window.showOpenFilePicker({
      multiple: false,
      types: [{description:"CSV", accept:{"text/csv":[".csv"]}}]
    });
    await attach(h);
  }catch(e){
    if(e.name !== "AbortError") fsStatus("link failed: " + e.message, "#d62728");
  }
}
async function newFile(){
  if(!HAS_FS){ fsStatus("browser cannot write files - use Export", "#d62728"); return; }
  try{
    const h = await window.showSaveFilePicker({
      suggestedName: "geocode_review.csv",
      types: [{description:"CSV", accept:{"text/csv":[".csv"]}}]
    });
    await attach(h);
  }catch(e){
    if(e.name !== "AbortError") fsStatus("create failed: " + e.message, "#d62728");
  }
}
async function grant(){
  if(!fileHandle) return;
  const p = await fileHandle.requestPermission({mode:"readwrite"});
  if(p === "granted"){
    document.getElementById("grantBtn").style.display = "none";
    await loadFromFile(); await saveNow();
  } else {
    fsStatus("permission denied", "#d62728");
  }
}
async function tryReconnect(){
  if(!HAS_FS){
    document.getElementById("linkBtn").disabled = true;
    document.getElementById("newBtn").disabled = true;
    fsStatus("browser storage only - use Export");
    return;
  }
  let h = null;
  try { h = await idbGet("handle"); } catch(e) { h = null; }
  if(!h){ fsStatus("not linked - click Link CSV"); return; }
  fileHandle = h;
  const p = await h.queryPermission({mode:"readwrite"});
  if(p === "granted"){
    const added = await loadFromFile();
    fsStatus("linked" + (added ? ", " + added + " loaded" : ""), "#2ca02c");
  } else {
    document.getElementById("grantBtn").style.display = "";
    fsStatus("linked - click Reconnect file to resume saving", "#8a6d00");
  }
}
document.addEventListener("keydown", e => {
  if(e.key==="ArrowRight") go(1);
  else if(e.key==="ArrowLeft") go(-1);
  else if(e.key==="1") setV("o", true);
  else if(e.key==="2") setV("o", false);
  else if(e.key==="3") setV("d", true);
  else if(e.key==="4") setV("d", false);
  else if(e.key==="5") setV("r", false);
  else if(e.key==="6") setV("r", true);
});
render();
tryReconnect();
</script></body></html>
"""

seed = {k: bool(as_verdict(v)) for k, v in existing.items() if as_verdict(v) is not None}
html_out = (REVIEW_HTML_TEMPLATE
            .replace("__ITEMS__", json.dumps(queue, separators=(",", ":")))
            .replace("__SEED__", json.dumps(seed, separators=(",", ":"))))

os.makedirs(os.path.dirname(REVIEW_HTML_FILE), exist_ok=True)
with open(REVIEW_HTML_FILE, "w") as f:
    f.write(html_out)
size_mb = os.path.getsize(REVIEW_HTML_FILE) / (1024 * 1024)
print(f"Wrote {REVIEW_HTML_FILE} ({size_mb:.2f} MB)")
print(f"  {len(queue):,} screens, {len(seed):,} verdicts pre-loaded")
print("\nOpen it in a browser, press 'Link CSV' and choose")
print(f"  {GEOCODE_REVIEW_FILE}")
print("so each verdict is saved as you click; then run section 5.")
print("Use 'New CSV' only if that file does not exist yet; it opens a Save")
print("dialog that can overwrite a file.")
print("Without linking, press 'Export decisions' when done and run section 4 first.")
if size_mb > 25:
    print("  NOTE: large file. Set INCLUDE_ROUTE_GEOMETRY = False or raise "
          "ROUTE_SIMPLIFY_DEG to shrink it.")

In [ ]:
# =========================================================================
# 4) FOLD AN EXPORTED DECISIONS FILE INTO THE SIDE FILE
#    Only needed after "Export decisions"; with a linked CSV it does nothing.
# =========================================================================
if not os.path.exists(REVIEW_EXPORT_FILE):
    print(f"No export file at {REVIEW_EXPORT_FILE}")
    if os.path.exists(GEOCODE_REVIEW_FILE):
        n_existing = len(pd.read_csv(GEOCODE_REVIEW_FILE))
        print(f"  {os.path.basename(GEOCODE_REVIEW_FILE)} already holds "
              f"{n_existing:,} verdicts, so there is nothing to fold in.")
        print("  Continue to section 5.")
    else:
        print(f"  {os.path.basename(GEOCODE_REVIEW_FILE)} does not exist either.")
        print("  Either press 'Link CSV' in the reviewer and choose that path, or")
        print("  press 'Export decisions' and point REVIEW_EXPORT_FILE at the download.")
else:
    new_rows = pd.read_csv(REVIEW_EXPORT_FILE)
    print(f"Export contains {len(new_rows):,} verdicts")

    old_rows = (pd.read_csv(GEOCODE_REVIEW_FILE) if os.path.exists(GEOCODE_REVIEW_FILE)
                else pd.DataFrame(columns=new_rows.columns))
    if "reviewed_utc" not in old_rows.columns:
        old_rows["reviewed_utc"] = pd.NA

    # Where the export and the side file overlap, the export wins.
    combined = pd.concat([old_rows, new_rows], ignore_index=True)
    review_df = save_review(combined.to_dict("records"), GEOCODE_REVIEW_FILE)
    print(review_df["verdict"].value_counts().to_string())


In [ ]:
# =========================================================================
# 5) APPLY THE VERDICTS TO THE ANALYSIS AND OD FILES
# =========================================================================
verdicts = load_review(GEOCODE_REVIEW_FILE)

# a) the manifest layer
manifests = gpd.read_file(MANURE_ANALYSIS_FILE)
olat = pd.to_numeric(manifests[ORIG_LAT_COL], errors="coerce")
olon = pd.to_numeric(manifests[ORIG_LON_COL], errors="coerce")
dlat = pd.to_numeric(manifests[DEST_LAT_COL], errors="coerce")
dlon = pd.to_numeric(manifests[DEST_LON_COL], errors="coerce")
oaddr = manifests[ORIG_ADDR_COL] if ORIG_ADDR_COL in manifests.columns else pd.Series("", index=manifests.index)
daddr = manifests[DEST_ADDR_COL] if DEST_ADDR_COL in manifests.columns else pd.Series("", index=manifests.index)

def verdict_series(side, addr, lat, lon):
    keys = [geocode_key(side, a, y, x) for a, y, x in zip(addr, lat, lon)]
    # Nullable boolean so GeoJSON receives true/false rather than a string
    return pd.Series([as_verdict(verdicts.get(k)) if k is not None else None for k in keys],
                     index=manifests.index, dtype="object").astype("boolean")

manifests[ORIG_OK_COL] = verdict_series("origin", oaddr, olat, olon)
manifests[DEST_OK_COL] = verdict_series("destination", daddr, dlat, dlon)

rkeys = [route_key(a, b, c, d) for a, b, c, d in zip(olat, olon, dlat, dlon)]
manifests[ROUTE_OK_COL] = pd.Series(
    [as_verdict(verdicts.get(k)) if k is not None else None for k in rkeys],
    index=manifests.index, dtype="object").astype("boolean")
# Each wrong verdict sets its own validation reason, in this order, on manifests
# that routing left valid. Reasons set by an earlier run of this section are
# cleared first.
REVIEW_REASONS = [
    (ORIG_OK_COL,  REASON_BAD_ORIGIN),
    (DEST_OK_COL,  REASON_BAD_DEST),
    (ROUTE_OK_COL, REASON_ROUTE_WRONG),
]

if VALID_COL in manifests.columns and REASON_COL in manifests.columns:
    owned = manifests[REASON_COL].astype("object").isin([r for _, r in REVIEW_REASONS])
    manifests.loc[owned, VALID_COL] = True
    manifests.loc[owned, REASON_COL] = None
    print(f"\n  cleared {int(owned.sum()):,} reason(s) assigned by a previous review run")

    for verdict_col, reason in REVIEW_REASONS:
        still_valid = manifests[VALID_COL].astype("boolean").fillna(False)
        hit = manifests[verdict_col].eq(False).fillna(False) & still_valid
        manifests.loc[hit, VALID_COL] = False
        manifests.loc[hit, REASON_COL] = reason
        print(f"  {reason:<26}{int(hit.sum()):>6,} manifests")
    print(f"  {'valid route = True':<26}{int(manifests[VALID_COL].astype(bool).sum()):>6,} manifests")
else:
    print(f"  {VALID_COL} / {REASON_COL} not present; run analysis/Routing.ipynb first")

manifests.to_file(MANURE_ANALYSIS_FILE, driver="GeoJSON")
print(f"\n{os.path.basename(MANURE_ANALYSIS_FILE)}")
print(f"  {ORIG_OK_COL}: {manifests[ORIG_OK_COL].notna().sum():,} judged "
      f"({(manifests[ORIG_OK_COL] == False).sum():,} wrong)")
print(f"  {DEST_OK_COL}: {manifests[DEST_OK_COL].notna().sum():,} judged "
      f"({(manifests[DEST_OK_COL] == False).sum():,} wrong)")
print(f"  {ROUTE_OK_COL}: {manifests[ROUTE_OK_COL].notna().sum():,} judged "
      f"({(manifests[ROUTE_OK_COL] == False).sum():,} flagged as bad routes)")

# b) the OD files, joined on RouteID; the verdicts are added as attributes.
od_cols = [ORIG_OK_COL, DEST_OK_COL, ROUTE_OK_COL, VALID_COL, REASON_COL]
by_route = manifests.set_index("RouteID")[[c for c in od_cols if c in manifests.columns]]
for path in (OD_LINES_FILE, OD_PAIRS_FILE):
    if not os.path.exists(path):
        print(f"\n  Skipping {os.path.basename(path)} (not found)")
        continue
    gdf = gpd.read_file(path)
    rid = pd.to_numeric(gdf["RouteID"], errors="coerce")
    for col in by_route.columns:
        mapped = rid.map(by_route[col])
        gdf[col] = mapped.astype("boolean") if col != REASON_COL else mapped.astype("object")
    gdf.to_file(path, driver="GeoJSON")
    print(f"\n{os.path.basename(path)}  ({len(gdf):,} features)")
    print(f"  {ORIG_OK_COL}: {gdf[ORIG_OK_COL].notna().sum():,} judged "
          f"({(gdf[ORIG_OK_COL] == False).sum():,} wrong)")
    print(f"  {DEST_OK_COL}: {gdf[DEST_OK_COL].notna().sum():,} judged "
          f"({(gdf[DEST_OK_COL] == False).sum():,} wrong)")
    print(f"  {ROUTE_OK_COL}: {gdf[ROUTE_OK_COL].notna().sum():,} judged "
          f"({(gdf[ROUTE_OK_COL] == False).sum():,} flagged as bad routes)")

print("\n REVIEW APPLIED")
print(" Re-running analysis/Routing.ipynb overwrites these files; run this "
      "section again afterwards to re-apply the verdicts from the side file.")